
# Allen–Cahn: traduzione MATLAB → Python


> **Versione commentata e riorganizzata.**  
> Il codice è stato diviso in blocchi più piccoli e i Markdown spiegano in particolare
> come vengono trattati i parametri $\alpha$ e $\mu$ nelle fasi di training e test.
> La logica numerica del notebook originale è stata mantenuta.

## Mappa completa del workflow

Il notebook segue un processo **offline-online**. L'obiettivo non e risolvere una sola simulazione, ma costruire modelli ridotti riutilizzabili per diversi valori dei parametri $\alpha$ e $\mu$.

### Fase 1 - Modello completo

Si parte dall'equazione semi-discreta di Allen-Cahn

$$
\dot y=\alpha Ly+\mu(y-y^3),
$$

con $y(t)\in\mathbb{R}^{576}$. Il FOM viene integrato nel tempo con Eulero implicito e Newton.

### Fase 2 - Generazione dei dati parametrici

Si scelgono nove coppie di training:

$$
\alpha\in\{0.01,0.1,1\},
\qquad
\mu\in\{1,6,11\}.
$$

Per ciascuna coppia si esegue una simulazione FOM e si salvano gli snapshot. Tutti gli snapshot vengono concatenati in $Y_{\mathrm{train}}$.

### Fase 3 - Costruzione dei ROM

Dagli stessi dati si costruiscono tre modelli:

- **POD-Galerkin:** comprime lo stato mediante SVD e proiezione di Galerkin;
- **POD-DEIM:** aggiunge DEIM per comprimere la valutazione del termine cubico;
- **Lift & Learn:** introduce $z=y^2$, riduce $(y,z)$ e apprende la dinamica mediante Operator Inference.

### Fase 4 - Nuovo punto parametrico

Dopo il training viene richiesto

$$
(\alpha^*,\mu^*)=(0.2,8).
$$

Le basi e i coefficienti non vengono ricalcolati. I nuovi valori entrano direttamente nelle equazioni ridotte.

### Fase 5 - Verifica

Si calcola anche il FOM nel punto $(0.2,8)$ e si confrontano le traiettorie mediante errore relativo, runtime e speedup.

In forma compatta:

$$
\boxed{
\text{PDE/FOM}
\to
\text{9 simulazioni}
\to
\text{snapshot}
\to
\text{training ROM}
\to
(0.2,8)
\to
\text{simulazione online}
\to
\text{errore e speedup}
}
$$


### Librerie e impostazioni numeriche

Questa cella prepara l'ambiente necessario per l'intero notebook.

- `time` viene usato per misurare i tempi di esecuzione di FOM e ROM.
- `warnings` permette di segnalare l'eventuale mancata convergenza del metodo di Newton.
- `numpy` gestisce vettori, matrici, norme, prodotti e SVD.
- `pandas` viene usato per costruire le tabelle riassuntive.
- `matplotlib.pyplot` genera i grafici.
- `scipy.sparse` costruisce Laplaciano, identità e Jacobiani sparsi.
- `scipy.sparse.linalg` risolve i sistemi lineari sparsi all'interno di Newton.
- `display` mostra i `DataFrame` in forma tabellare nel notebook.

Infine,

```text
np.set_printoptions(precision=6, suppress=True)
```

imposta la visualizzazione degli array a 6 cifre decimali e sopprime, quando possibile, la notazione scientifica. Questa istruzione modifica soltanto la **stampa** dei valori, non i calcoli.


In [ ]:

import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scipy.sparse.linalg as spla

from IPython.display import display

np.set_printoptions(precision=6, suppress=True)


## Funzioni locali

### 1. Risolutore non lineare e FOM parametrico

La prima parte contiene il **modello completo (FOM)** dell'equazione di Allen–Cahn.

Il modello semi-discreto utilizzato nel notebook è

$$
\dot y
=
\alpha Ly+\mu(y-y^3),
$$

dove:

- $y(t)\in\mathbb{R}^N$ è lo stato discretizzato;
- $L$ è il Laplaciano discreto;
- $\alpha$ moltiplica il termine di **diffusione**;
- $\mu$ moltiplica il termine di **reazione non lineare**.

Il punto essenziale per il **caso parametrico** è questo:

> `alpha` e `mu` sono argomenti delle funzioni di simulazione.  
> Non vengono ricavati facendo una media o un'interpolazione delle soluzioni di training.

Quindi, quando più avanti useremo `alpha = 0.2` e `mu = 8`, la stessa equazione verrà semplicemente risolta con quei due nuovi coefficienti.

Con Eulero implicito, a ogni passo temporale si cerca $y^{n+1}$ tale che

$$
y^{n+1}
-y^n
-\Delta t
\left[
\alpha L y^{n+1}
+
\mu\left(y^{n+1}-(y^{n+1})^3\right)
\right]
=0.
$$

Poiché compare il termine cubico, il sistema è non lineare e viene risolto con Newton.
#### Perche questo e il punto di partenza

Prima di costruire un modello ridotto serve un modello completo affidabile che produca i dati di riferimento. Il FOM descrive l'evoluzione di tutte le $N=576$ variabili spaziali e viene usato in due modi:

1. **offline**, per generare gli snapshot di training;
2. **come riferimento**, per misurare l'errore dei ROM.

Il flusso di un singolo passo temporale FOM e quindi

$$
y^n \;\longrightarrow\; F(y^{n+1})=0 \;\longrightarrow\; \text{Newton} \;\longrightarrow\; y^{n+1}.
$$

Il motivo per cui Newton e necessario e la presenza del termine cubico $(y^{n+1})^3$: il passo implicito non e un semplice sistema lineare. In pratica, a ogni tempo il codice costruisce il residuo `F_fun`, il Jacobiano `J_fun` e aggiorna la soluzione finche la correzione di Newton e sufficientemente piccola.

#### Dove entrano i parametri

I parametri non sono etichette associate alla soluzione: compaiono direttamente nell'equazione che viene risolta. Se si chiama la funzione con $(\alpha,\mu)=(0.1,6)$, il FOM risolve

$$
\dot y = 0.1\,Ly + 6(y-y^3),
$$

mentre con $(\alpha,\mu)=(1,11)$ risolve

$$
\dot y = Ly + 11(y-y^3).
$$

Questa osservazione sara importante anche nella fase online dei ROM: il nuovo punto parametrico viene inserito nella dinamica, non ottenuto facendo la media di traiettorie gia note.


In [ ]:
def solve_newton(x0, F_fun, J_fun):
    """Metodo di Newton equivalente alla funzione MATLAB solve_newton."""
    tol = 1e-5
    err = 1.0
    iteration = 0
    max_iter = 100
    x_new = np.asarray(x0, dtype=float).reshape(-1).copy()

    while err > tol and iteration < max_iter:
        F_val = np.asarray(F_fun(x_new)).reshape(-1)
        J_val = J_fun(x_new)
        # MATLAB: delta_x = J_val \ F_val
        delta_x = spla.spsolve(J_val, F_val)
        x_new = x_new - delta_x
        err = np.linalg.norm(delta_x, ord=np.inf)
        iteration += 1

    if iteration == max_iter:
        warnings.warn('Newton non converge!', RuntimeWarning)

    return x_new

def simulate_allen_fom_implicit(L, y0, dt, times, save, snapshot_times, alpha, mu):
    """FOM Allen–Cahn parametrico con Eulero implicito e Newton.

    I parametri alpha e mu NON vengono interpolati da casi già simulati:
    vengono passati direttamente alla funzione e inseriti nell'equazione
        y_t = alpha * L y + mu * (y - y^3).
    Per questo la stessa routine può essere usata sia sui punti di training
    sia su una nuova coppia parametrica, ad esempio (alpha, mu) = (0.2, 8).
    """
    N = len(y0)
    state = np.asarray(y0, dtype=float).reshape(-1).copy()

    snapshot_matrix = np.zeros((N, len(snapshot_times)), dtype=float)
    snapshot_matrix[:, 0] = state

    I = sp.eye(N, format='csc')
    # Parte costante del Jacobiano del passo implicito:
    # J_const = I - dt*alpha*L - dt*mu*I
    # Qui si vede già dove entrano DIRETTAMENTE i due parametri.
    J_const = (I - dt * alpha * L - dt * mu * I).tocsc()

    save_counter = 0
    tic = time.perf_counter()

    for step in range(1, len(times)):
        # Residuo del passo di Eulero implicito:
        # x - state - dt*[alpha*L*x + mu*(x - x^3)] = 0
        def F_fun(x):
            return J_const @ x + dt * mu * (x ** 3) - state

        # Jacobiano del residuo rispetto alla nuova soluzione x.
        def J_fun(x):
            return J_const + sp.diags(3.0 * dt * mu * (x ** 2), offsets=0, format='csc')

        state = solve_newton(state, F_fun, J_fun)

        if step % save == 0:
            save_counter += 1
            snapshot_matrix[:, save_counter] = state

    runtime = time.perf_counter() - tic
    return snapshot_matrix[:, :save_counter + 1], runtime

### 2. Funzioni ausiliarie per i termini quadratici

Il modello Lift & Learn usa uno stato ridotto che contiene anche la variabile sollevata

$$
z=y^2.
$$

Per costruire un modello quadratico servono quindi tutti i monomi

$$
x_i x_j,\qquad i\le j.
$$

Le due funzioni seguenti costruiscono questi monomi per un singolo vettore (`qvec`) o per molte colonne (`qmat`).

In [ ]:
def qvec(x):
    """Monomi quadratici xi*xj senza ripetizioni, con i <= j."""
    x = np.asarray(x).reshape(-1)
    r = len(x)
    q = np.zeros(r * (r + 1) // 2, dtype=float)

    row = 0
    for i in range(r):
        for j in range(i, r):
            q[row] = x[i] * x[j]
            row += 1

    return q

def qmat(X):
    """Applica qvec colonna per colonna: shape r(r+1)/2 x K."""
    X = np.asarray(X)
    r, K = X.shape
    Q = np.zeros((r * (r + 1) // 2, K), dtype=float)

    row = 0
    for i in range(r):
        for j in range(i, r):
            Q[row, :] = X[i, :] * X[j, :]
            row += 1

    return Q

### 3. Training POD e POD-DEIM

Queste funzioni costruiscono le basi ridotte usando la matrice globale degli snapshot.

La matrice di training contiene snapshot provenienti da **tutte le coppie parametriche di training**.  
Per questo la base POD è una **base globale parametrica**: deve rappresentare soluzioni generate con diversi valori di $\alpha$ e $\mu$.

Attenzione però:

- la base POD viene appresa dai dati;
- i valori nuovi di $\alpha$ e $\mu$ **non** vengono appresi o interpolati qui;
- i parametri entreranno successivamente, durante la simulazione online del ROM.
#### Perche si usa una base globale

Una base POD costruita da una sola simulazione descriverebbe soprattutto quella specifica traiettoria. Qui invece si vuole un ROM **parametrico**, cioe utilizzabile per una famiglia di valori di $\alpha$ e $\mu$. Per questo la SVD viene applicata alla matrice che contiene gli snapshot di tutte le nove simulazioni FOM.

Il processo offline e

$$
(\alpha_i,\mu_i)
\;\longrightarrow\;
Y_i
\;\longrightarrow\;
Y_{\mathrm{train}}=[Y_1\;Y_2\;\cdots\;Y_9]
\;\longrightarrow\;
\mathrm{SVD}
\;\longrightarrow\;
\Psi.
$$

La base $\Psi$ cerca quindi di catturare le strutture spaziali dominanti dell'intera famiglia di soluzioni osservate nel training.

Per POD-DEIM si esegue una seconda compressione sugli snapshot del termine cubico. DEIM sceglie pochi punti spaziali rappresentativi in cui valutare la non linearita, con l'obiettivo di evitare che il costo online dipenda ancora fortemente dalla dimensione completa $N$.


In [ ]:
def deim_indices(U):
    """Algoritmo greedy DEIM per la selezione dei punti spaziali."""
    U = np.asarray(U)
    indices = [int(np.argmax(np.abs(U[:, 0])))]

    for i in range(1, U.shape[1]):
        # MATLAB: c = U(indices, 1:i-1) \ U(indices, i)
        selected_matrix = U[np.ix_(indices, np.arange(i))]
        selected_rhs = U[indices, i]
        c = np.linalg.solve(selected_matrix, selected_rhs)

        residual = U[:, i] - U[:, :i] @ c
        new_index = int(np.argmax(np.abs(residual)))
        indices.append(new_index)

    return np.asarray(indices, dtype=int)

def train_allen_pod(L, snapshots, r, m):
    """Costruisce una base POD globale e gli oggetti POD-DEIM.

    La base viene ottenuta usando snapshot provenienti da TUTTE le coppie
    (alpha, mu) di training. La base quindi riassume una famiglia parametrica
    di soluzioni, ma in questa fase non viene interpolato alcun parametro.
    """
    # POD globale dello stato: gli snapshot di tutte le simulazioni
    # parametriche sono già concatenati nella stessa matrice.
    U, sv_linear, _ = np.linalg.svd(snapshots, full_matrices=False)
    Psi = U[:, :min(r, U.shape[1])]
    # Operatore diffusivo ridotto. Il fattore alpha verrà applicato ONLINE,
    # quando si simula una specifica coppia parametrica.
    Lr = np.asarray(Psi.T @ (L @ Psi))

    # Basi POD non lineari
    Un, sv_nonlinear, _ = np.linalg.svd(snapshots ** 3, full_matrices=False)
    Un = Un[:, :min(m, Un.shape[1])]

    # Selezione DEIM
    deim_idx = deim_indices(Un)

    # MATLAB: nonlinear_projection = (Psi' * Un) / Un(deim_idx, :)
    # A / B = A @ inv(B); usiamo solve per maggiore stabilità numerica.
    A = Psi.T @ Un
    B = Un[deim_idx, :]
    nonlinear_projection = np.linalg.solve(B.T, A.T).T

    sampled_Psi = Psi[deim_idx, :]

    return (
        Psi,
        Lr,
        sv_linear,
        nonlinear_projection,
        sampled_Psi,
        deim_idx,
        sv_nonlinear,
    )

### 4. Costruzione dei dati parametrici Lift & Learn

Per Lift & Learn si introduce il lifting

$$
z=y^2
$$

e si costruisce lo stato ridotto complessivo

$$
x=
\begin{bmatrix}
a_y\\
a_z
\end{bmatrix}.
$$

Per ogni snapshot di training viene conservata anche la coppia $(\alpha,\mu)$ che lo ha generato.

Dopo aver costruito i monomi quadratici $q(x)$, il regressore parametrico è

$$
\Theta(x,\alpha,\mu)
=
\begin{bmatrix}
\alpha x\\
\mu x\\
\alpha q(x)\\
\mu q(x)
\end{bmatrix}.
$$

L'Operator Inference cerca quindi una matrice di coefficienti $C$ tale che

$$
\dot x
\approx
C\,\Theta(x,\alpha,\mu).
$$

Questa è la parte più importante per capire **come il Lift & Learn tratta un parametro mai visto**.
#### Perche si introduce il lifting

La dinamica originale contiene $y^3$. Introducendo

$$
z=y^2,
$$

si puo riscrivere il termine cubico come prodotto $yz$. In questo modo la struttura lifted puo essere rappresentata usando termini lineari e quadratici, che sono adatti all'Operator Inference impiegato nel notebook.

Lo stato ridotto lifted e

$$
x=
\begin{bmatrix}
a_y\\
a_z
\end{bmatrix},
$$

dove $a_y$ sono le coordinate POD di $y$ e $a_z$ quelle di $z=y^2$.

Per imparare una **dinamica** non basta conoscere $x$: serve anche la sua derivata. Per ogni snapshot il codice valuta

$$
\dot y=\alpha Ly+\mu(y-y^3),
$$

e, usando la regola della catena,

$$
\dot z=2y\dot y.
$$

Dopo la proiezione si ottiene il dataset di coppie $(x,\dot x)$.

#### Perche compaiono $\alpha x$, $\mu x$, $\alpha q(x)$ e $\mu q(x)$

La dinamica di Allen-Cahn separa naturalmente una parte moltiplicata da $\alpha$ e una parte moltiplicata da $\mu$. Il modello appreso deve conservare questa dipendenza. Per questo, per ogni snapshot, si costruiscono le feature

$$
\Theta(x,\alpha,\mu)=
\begin{bmatrix}
\alpha x\\
\mu x\\
\alpha q(x)\\
\mu q(x)
\end{bmatrix}.
$$

Qui $q(x)$ contiene tutti i monomi quadratici unici $x_i x_j$ con $i\le j$. L'Operator Inference apprende quindi una matrice $C$ tale che

$$
\dot x\approx C\,\Theta(x,\alpha,\mu).
$$

Questa e la ragione per cui, una volta concluso il training, lo stesso modello puo essere valutato con una nuova coppia parametrica: basta ricostruire le stesse feature usando i nuovi valori di $\alpha$ e $\mu$.


In [ ]:
def build_allen_lift_data(L, alphas, mus, Y_all, num_snaps_per_sim, ry, rz):
    """Costruisce i dati parametrici per Lift & Learn / Operator Inference.

    Ogni snapshot mantiene associati i valori (alpha, mu) della simulazione
    da cui proviene. Dopo la riduzione, le feature vengono costruite come

        [alpha*x, mu*x, alpha*q(x), mu*q(x)]

    così il modello imparato può essere valutato anche per nuovi valori
    dei parametri senza interpolare direttamente le traiettorie FOM.
    """
    # LIFTING: T(y) = y.^2
    Z_all = Y_all ** 2

    # POD separata per stato originale e stato sollevato
    Uy, sv_y, _ = np.linalg.svd(Y_all, full_matrices=False)
    Psi_y = Uy[:, :ry]

    Uz, sv_z, _ = np.linalg.svd(Z_all, full_matrices=False)
    Psi_z = Uz[:, :rz]

    K_tot = Y_all.shape[1]
    X = np.zeros((ry + rz, K_tot), dtype=float)
    X_dot = np.zeros((ry + rz, K_tot), dtype=float)
    alpha_row = np.zeros(K_tot, dtype=float)
    mu_row = np.zeros(K_tot, dtype=float)

    for k in range(len(alphas)):
        a_k = alphas[k]
        m_k = mus[k]

        col_start = k * num_snaps_per_sim
        col_end = (k + 1) * num_snaps_per_sim
        states = Y_all[:, col_start:col_end]

        for j in range(num_snaps_per_sim):
            idx = col_start + j

            # Ricostruzione sullo spazio pieno filtrata dalla POD
            proj_y = Psi_y @ (Psi_y.T @ states[:, j])
            dy = a_k * (L @ proj_y) + m_k * (proj_y - proj_y ** 3)
            dz = 2.0 * proj_y * dy

            X[:, idx] = np.concatenate((
                Psi_y.T @ states[:, j],
                Psi_z.T @ (states[:, j] ** 2),
            ))

            X_dot[:, idx] = np.concatenate((
                Psi_y.T @ dy,
                Psi_z.T @ dz,
            ))

            # Salviamo i parametri associati a QUESTO snapshot.
            # Serviranno poco sotto per costruire le feature parametriche.
            alpha_row[idx] = a_k
            mu_row[idx] = m_k

    Q = qmat(X)

    # FEATURE PARAMETRICHE.
    # Se x è lo stato ridotto e q(x) contiene i monomi quadratici,
    # costruiamo: alpha*x, mu*x, alpha*q(x), mu*q(x).
    # È questo il passaggio che rende l'Operator Inference parametrico.
    alpha_lin = X * alpha_row[None, :]
    mu_lin = X * mu_row[None, :]
    alpha_quad = Q * alpha_row[None, :]
    mu_quad = Q * mu_row[None, :]

    F = np.vstack((alpha_lin, mu_lin, alpha_quad, mu_quad))

    # Classificazione dei monomi quadratici
    num_q = Q.shape[0]
    is_yy = np.zeros(num_q, dtype=bool)
    is_yz = np.zeros(num_q, dtype=bool)
    is_zz = np.zeros(num_q, dtype=bool)

    row = 0
    for i in range(ry + rz):
        for j in range(i, ry + rz):
            if i < ry and j < ry:
                is_yy[row] = True
            elif i >= ry and j >= ry:
                is_zz[row] = True
            else:
                is_yz[row] = True
            row += 1

    rtot = ry + rz
    idx_alpha_lin = np.arange(0, rtot)
    idx_mu_lin = np.arange(rtot, 2 * rtot)
    idx_alpha_quad = np.arange(2 * rtot, 2 * rtot + num_q)
    idx_mu_quad = np.arange(2 * rtot + num_q, 2 * rtot + 2 * num_q)

    # Maschera dei coefficienti ammessi
    masks = np.zeros((rtot, F.shape[0]), dtype=bool)

    for i in range(rtot):
        if i < ry:
            # Equazioni dello stato originale
            masks[i, idx_alpha_lin[:ry]] = True
            masks[i, idx_mu_lin[:ry]] = True
            masks[i, idx_mu_quad[is_yz]] = True
        else:
            # Equazioni dello stato sollevato
            masks[i, idx_alpha_quad[is_yy]] = True
            masks[i, idx_mu_lin[ry:]] = True
            masks[i, idx_mu_quad[is_zz]] = True

    return Psi_y, Psi_z, sv_y, sv_z, F, X_dot, masks

def OpInf(F, X_dot, masks, reg):
    """Operator Inference con maschera e regolarizzazione Tikhonov."""
    D = F.T
    coefficients = np.zeros((X_dot.shape[0], F.shape[0]), dtype=float)

    for i in range(X_dot.shape[0]):
        active_cols = np.flatnonzero(masks[i, :])
        D_act = D[:, active_cols]

        # Normalizzazione colonne
        col_norms = np.sqrt(np.sum(D_act ** 2, axis=0))
        col_norms[col_norms == 0.0] = 1.0
        D_scaled = D_act / col_norms[None, :]

        # Regolarizzazione
        num_act = D_scaled.shape[1]
        augmented_matrix = np.vstack((
            D_scaled,
            np.sqrt(reg) * np.eye(num_act),
        ))
        augmented_target = np.concatenate((
            X_dot[i, :],
            np.zeros(num_act),
        ))

        # MATLAB backslash su sistema rettangolare -> least squares
        sol, *_ = np.linalg.lstsq(augmented_matrix, augmented_target, rcond=None)
        coefficients[i, active_cols] = sol / col_norms

    return coefficients

def fit_allen_lift_coefficients(F, X_dot, masks, regularization):
    coefficients = OpInf(F, X_dot, masks, regularization)
    training_residual = (
        np.linalg.norm(coefficients @ F - X_dot, ord='fro')
        / np.linalg.norm(X_dot, ord='fro')
    )
    return coefficients, training_residual

### 5. Simulazione online dei tre ROM

Le tre funzioni seguenti vengono usate **dopo il training**.

In questa fase le basi e/o i coefficienti sono già fissati.  
Per una nuova coppia $(\alpha^*,\mu^*)$, ad esempio

$$
(\alpha^*,\mu^*)=(0.2,8),
$$

non si ricostruisce il training e non si interpola tra le nove traiettorie note.

Si inseriscono invece direttamente i nuovi parametri nelle rispettive dinamiche ridotte:

**POD-Galerkin**
$$
\dot a
=
\alpha^* L_r a
+
\mu^*
\left[
a-\Psi^T(\Psi a)^3
\right].
$$

**POD-DEIM**
$$
\dot a
=
\alpha^* L_r a
+
\mu^*
\left[
a-P_{\mathrm{DEIM}}(S^T\Psi a)^3
\right].
$$

**Lift & Learn**
$$
\dot x
=
C
\begin{bmatrix}
\alpha^*x\\
\mu^*x\\
\alpha^*q(x)\\
\mu^*q(x)
\end{bmatrix}.
$$
#### Cosa significa "online" nella pratica

La fase **offline** ha gia prodotto gli oggetti costosi:

- la base POD $\Psi$ e l'operatore ridotto $L_r$;
- la base della non linearita e i punti DEIM;
- le basi lifted $\Psi_y$, $\Psi_z$ e i coefficienti $C$ di Operator Inference.

Nella fase **online** arriva una nuova richiesta, per esempio

$$
(\alpha^*,\mu^*)=(0.2,8).
$$

Non si rifanno le nove simulazioni FOM, non si ricalcola la SVD e non si interpola tra le traiettorie di training. Si integra direttamente la nuova dinamica ridotta con quei coefficienti parametrici.

Per POD, a ogni passo temporale:

$$
a_k
\;\longrightarrow\;
y_k\approx\Psi a_k
\;\longrightarrow\;
\dot a_k
\;\longrightarrow\;
a_{k+1}=a_k+\Delta t\,\dot a_k.
$$

Per Lift & Learn il flusso e:

$$
a_y^k
\longrightarrow
a_z^k
\longrightarrow
x^k
\longrightarrow
q(x^k)
\longrightarrow
\Theta(x^k,0.2,8)
\longrightarrow
C\Theta
\longrightarrow
\dot x^k.
$$

In particolare, nel punto di test le feature diventano

$$
\Theta_{\mathrm{test}}=
\begin{bmatrix}
0.2x\\
8x\\
0.2q(x)\\
8q(x)
\end{bmatrix}.
$$

Questo e il significato pratico della **generalizzazione parametrica**: il ROM usa una legge dinamica ridotta costruita offline e la valuta con parametri che non appartengono alle nove coppie di training.


In [ ]:
def simulate_allen_pod(Psi, Lr, y0, alpha, mu, dt, times, save):
    """ROM POD-Galerkin parametrico con Eulero esplicito.

    Online si usano direttamente i valori alpha e mu richiesti:
        da/dt = alpha*Lr*a + mu*(a - Psi.T@(Psi*a)^3).
    La base Psi è globale, ma i parametri della nuova simulazione entrano
    direttamente nella dinamica ridotta.
    """
    a_k = Psi.T @ y0
    ry = Psi.shape[1]

    A = np.zeros((ry, (len(times) - 1) // save + 1), dtype=float)
    A[:, 0] = a_k

    tic = time.perf_counter()
    save_counter = 0

    for step in range(1, len(times)):
        full_state = Psi @ a_k
        # Nuova dinamica ridotta per la coppia (alpha, mu) passata alla funzione.
        da = alpha * (Lr @ a_k) + mu * (a_k - Psi.T @ (full_state ** 3))
        a_k = a_k + dt * da

        if step % save == 0:
            save_counter += 1
            A[:, save_counter] = a_k

    runtime = time.perf_counter() - tic
    reconstructed_state = Psi @ A[:, :save_counter + 1]
    return reconstructed_state, runtime

def simulate_allen_deim(Psi, Lr, proj, s_Psi, y0, alpha, mu, dt, times, save):
    """ROM POD-DEIM parametrico con Eulero esplicito.

    Come nel POD, alpha e mu entrano direttamente nella dinamica online.
    DEIM cambia solo il modo in cui viene approssimato il termine cubico:
    invece di ricostruirlo su tutti i N nodi, lo valuta nei punti DEIM.
    """
    a_k = Psi.T @ y0
    ry = Psi.shape[1]

    A = np.zeros((ry, (len(times) - 1) // save + 1), dtype=float)
    A[:, 0] = a_k

    tic = time.perf_counter()
    save_counter = 0

    for step in range(1, len(times)):
        sampled_state = s_Psi @ a_k
        # alpha pesa la diffusione ridotta; mu pesa la reazione non lineare.
        da = alpha * (Lr @ a_k) + mu * (a_k - proj @ (sampled_state ** 3))
        a_k = a_k + dt * da

        if step % save == 0:
            save_counter += 1
            A[:, save_counter] = a_k

    runtime = time.perf_counter() - tic
    reconstructed_state = Psi @ A[:, :save_counter + 1]
    return reconstructed_state, runtime

def simulate_allen_lift(y0, Psi_y, Psi_z, coefficients, alpha, mu, dt, times, save_stride):
    """Simulazione online del modello Lift & Learn parametrico.

    Per ogni stato ridotto vengono costruite le feature
        [alpha*x, mu*x, alpha*q(x), mu*q(x)]
    usando i NUOVI valori di alpha e mu forniti alla simulazione.
    I coefficienti appresi durante il training trasformano poi queste feature
    nella derivata dello stato ridotto.
    """
    ry = Psi_y.shape[1]
    rz = Psi_z.shape[1]

    # Matrice per ricostruire esattamente il lifting quadratico proiettato
    M = np.zeros((rz, ry * (ry + 1) // 2), dtype=float)
    col = 0

    for i in range(ry):
        for j in range(i, ry):
            prod_vec = Psi_y[:, i] * Psi_y[:, j]
            if i < j:
                prod_vec = 2.0 * prod_vec
            M[:, col] = Psi_z.T @ prod_vec
            col += 1

    a_y = Psi_y.T @ y0
    A = np.zeros((ry, (len(times) - 1) // save_stride + 1), dtype=float)
    A[:, 0] = a_y

    tic = time.perf_counter()
    save_counter = 0

    for step in range(1, len(times)):
        q_y = qvec(a_y)
        a_z = M @ q_y
        state_all = np.concatenate((a_y, a_z))
        quad_all = qvec(state_all)

        # Feature parametriche valutate nel punto corrente.
        # Per il test (0.2, 8), ad esempio, diventano
        # [0.2*x, 8*x, 0.2*q(x), 8*q(x)].
        features = np.concatenate((
            alpha * state_all,
            mu * state_all,
            alpha * quad_all,
            mu * quad_all,
        ))

        # Operator Inference: i coefficienti appresi trasformano le feature
        # nella derivata del sistema lifted ridotto.
        full_deriv = coefficients @ features
        a_y = a_y + dt * full_deriv[:ry]

        if step % save_stride == 0:
            save_counter += 1
            A[:, save_counter] = a_y

    runtime = time.perf_counter() - tic
    solution = Psi_y @ A[:, :save_counter + 1]
    return solution, runtime

## Parametri, griglia spaziale e discretizzazione temporale

### Discretizzazione spaziale

Si discretizza il dominio bidimensionale con

$$
n=24
$$

nodi interni per direzione, quindi

$$
N=n^2=576.
$$

Il Laplaciano discreto $L$ viene costruito tramite differenze finite centrate e prodotti di Kronecker.

La condizione iniziale è

$$
y_0(x_1,x_2)
=
0.1\sin(\pi x_1)\sin(\pi x_2).
$$

Questa parte è **indipendente dai parametri** $\alpha$ e $\mu$: griglia, Laplaciano e condizione iniziale rimangono uguali per tutte le simulazioni.
#### Dal campo 2D al vettore numerico

La PDE vive su un dominio bidimensionale, ma il solutore lavora con vettori. La griglia $24\times24$ trasforma quindi il campo $y(x_1,x_2,t)$ in un vettore

$$
y(t)\in\mathbb{R}^{576}.
$$

Il Laplaciano continuo viene sostituito dal Laplaciano discreto $L$. Da questo momento la PDE e trattata numericamente come un grande sistema di equazioni differenziali ordinarie. Questa e la forma su cui lavorano sia il FOM sia i ROM.


In [ ]:
# ============================================================
# BLOCCO A — GRIGLIA SPAZIALE E OPERATORE DI LAPLACE
# ============================================================

n = 24
T = 2.0
N = n * n  # numero totale di gradi di libertà: 24^2 = 576

dx = 1.0 / (n + 1)
x = np.linspace(dx, 1.0 - dx, n)

# Laplaciano 1D con differenze finite centrate.
e = np.ones(n)
L1 = sp.spdiags(
    np.vstack((e, -2.0 * e, e)),
    [-1, 0, 1],
    n,
    n,
).tocsc() / dx**2

# Laplaciano 2D ottenuto mediante prodotti di Kronecker.
L = (
    sp.kron(sp.eye(n, format='csc'), L1, format='csc')
    + sp.kron(L1, sp.eye(n, format='csc'), format='csc')
).tocsc()

# Griglia 2D.
X1, X2 = np.meshgrid(x, x, indexing='xy')

# Condizione iniziale.
initial_field = 0.1 * np.sin(np.pi * X1) * np.sin(np.pi * X2)
y0 = initial_field.T.reshape(N, order='F')

### Discretizzazione temporale

Il FOM usa un passo temporale piccolo

$$
\Delta t=0.0004,
$$

ma non salva ogni singolo passo. Gli snapshot vengono memorizzati ogni

$$
\Delta t_{\mathrm{snap}}=0.04.
$$

Quindi `save` indica ogni quanti passi temporali viene salvato uno snapshot.
#### Perche snapshot e passi temporali sono diversi

Il FOM evolve con un passo molto piccolo $\Delta t=0.0004$ per mantenere l'accuratezza numerica, ma non e necessario memorizzare tutti gli stati intermedi. Gli snapshot vengono salvati ogni $0.04$, cioe ogni 100 passi FOM.

Quindi bisogna distinguere:

- **passo di integrazione**: serve a far avanzare la soluzione;
- **passo di snapshot**: decide quando conservare uno stato per il training e per i confronti.

Questa scelta riduce la memoria necessaria senza cambiare la dinamica numerica usata dal FOM.


In [ ]:
# ============================================================
# BLOCCO B — DISCRETIZZAZIONE TEMPORALE
# ============================================================

snapshot_dt = 0.04
dt = 0.0004

num_steps = int(np.ceil(T / dt))
times = np.linspace(0.0, T, num_steps + 1)

# Numero di passi FOM tra due snapshot consecutivi.
save = int(np.rint(snapshot_dt / dt))
snapshot_times = times[::save]

### Punto parametrico usato per verificare la generalizzazione

Il training verrà costruito con altre coppie $(\alpha,\mu)$.  
Qui definiamo invece il punto

$$
\boxed{\alpha_{\mathrm{test}}=0.2,\qquad \mu_{\mathrm{test}}=8}
$$

che non appartiene alla griglia di training.

Questo punto servirà a verificare se i modelli ridotti riescono a simulare una dinamica corrispondente a parametri **non presenti tra le nove simulazioni usate per costruire le basi/dati**.
#### Perche scegliere un punto non appartenente alla griglia

Le nove coppie di training servono a costruire il modello. Successivamente si vuole verificare se il ROM e capace di lavorare anche in un punto non usato per costruire la base o le feature. Per questo viene scelto

$$
(\alpha_{\mathrm{test}},\mu_{\mathrm{test}})=(0.2,8).
$$

L'idea non e chiedere al ROM di ricordare una traiettoria gia vista, ma verificare se la rappresentazione ridotta e la dipendenza parametrica appresa sono sufficienti a generare una nuova traiettoria coerente.


In [ ]:
# ============================================================
# BLOCCO C — PARAMETRI DEL CASO NON PRESENTE NEL TRAINING
# ============================================================

test_alpha = 0.2
test_mu = 8.0

print(f'N = {N}, dx = {dx:.6f}, passi temporali = {len(times)-1}')
print(f'snapshot ogni {save} passi -> {len(snapshot_times)} snapshot per simulazione')
print(f'punto parametrico di test: alpha={test_alpha:g}, mu={test_mu:g}')

## Dati di training: generazione snapshot FOM

### Griglia parametrica di training

Il training utilizza

$$
\alpha\in\{0.01,\;0.1,\;1\},
\qquad
\mu\in\{1,\;6,\;11\}.
$$

Il prodotto cartesiano genera $3\times3=9$ coppie:

$$
\begin{aligned}
&(0.01,1),\;(0.01,6),\;(0.01,11),\\
&(0.1,1),\;(0.1,6),\;(0.1,11),\\
&(1,1),\;(1,6),\;(1,11).
\end{aligned}
$$

Per **ogni coppia** viene risolto nuovamente il FOM

$$
\dot y=\alpha Ly+\mu(y-y^3).
$$

Quindi il training contiene informazioni su come cambia la soluzione quando cambiano i parametri.

Gli snapshot delle nove simulazioni vengono poi concatenati nella matrice

$$
Y_{\mathrm{train}}
=
\begin{bmatrix}
Y^{(1)} & Y^{(2)} & \cdots & Y^{(9)}
\end{bmatrix}.
$$

Questa matrice unica verrà usata per costruire le basi POD e Lift.

> Il test $(0.2,8)$ non viene inserito in questa matrice.
#### Cosa produce il ciclo sulle nove coppie

Per ogni coppia $(\alpha_k,\mu_k)$ viene eseguito nuovamente il FOM completo. Ogni simulazione produce una matrice di snapshot $Y_k$. I nove blocchi vengono poi concatenati:

$$
Y_{\mathrm{train}}=
\begin{bmatrix}
Y_1 & Y_2 & \cdots & Y_9
\end{bmatrix}.
$$

Questa e la banca dati comune usata per costruire sia la POD classica sia le basi del modello lifted. Il training parametrico non consiste quindi nel variare i parametri dentro una formula di interpolazione: consiste nel fornire al ROM esempi FOM provenienti da diverse dinamiche della stessa famiglia.


In [ ]:
# ============================================================
# BLOCCO A — DEFINIZIONE DELLA GRIGLIA PARAMETRICA DI TRAINING
# ============================================================

alpha_values = np.array([0.01, 0.1, 1.0])
mu_values = np.array([1.0, 6.0, 11.0])

# Costruiamo tutte le combinazioni (alpha, mu).
# reshape(..., order='F') mantiene lo stesso ordinamento del MATLAB originale.
A_grid, M_grid = np.meshgrid(alpha_values, mu_values, indexing='xy')
training_alphas = A_grid.reshape(-1, order='F')
training_mus = M_grid.reshape(-1, order='F')

num_sims = len(training_alphas)  # 3 x 3 = 9 simulazioni

print("Coppie parametriche di training:")
for k, (a, m) in enumerate(zip(training_alphas, training_mus), start=1):
    print(f"  {k:>2}: alpha={a:g}, mu={m:g}")

In [ ]:
# ============================================================
# BLOCCO B — ALLOCAZIONE DELLA MATRICE GLOBALE DEGLI SNAPSHOT
# ============================================================

num_snaps_per_sim = len(snapshot_times)

# Ogni blocco di colonne contiene la traiettoria di UNA coppia parametrica.
snapshot_matrix = np.zeros(
    (N, num_snaps_per_sim * num_sims),
    dtype=float,
)

fom_train_time = 0.0

In [ ]:
# ============================================================
# BLOCCO C — 9 SIMULAZIONI FOM DI TRAINING
# ============================================================

for k in range(num_sims):
    alpha_k = training_alphas[k]
    mu_k = training_mus[k]

    # La stessa funzione FOM viene richiamata con una nuova coppia (alpha_k, mu_k).
    # Non stiamo interpolando una soluzione precedente: risolviamo di nuovo
    # l'equazione di Allen–Cahn con quei coefficienti.
    states, rt = simulate_allen_fom_implicit(
        L, y0, dt, times, save, snapshot_times,
        alpha_k, mu_k
    )

    # Inseriamo gli snapshot della simulazione k nel blocco corretto di colonne.
    col_start = k * num_snaps_per_sim
    col_end = (k + 1) * num_snaps_per_sim
    snapshot_matrix[:, col_start:col_end] = states

    fom_train_time += rt

    print(
        f'FOM training {k+1:>2}/{num_sims} | '
        f'alpha={alpha_k:g}, mu={mu_k:g} | '
        f'{rt:.3f} s'
    )

print(f'\nTempo FOM totale per generazione training: {fom_train_time:.3f} s')

## Test FOM

### FOM di riferimento nel punto parametrico non visto

Ora calcoliamo il **ground truth** per

$$
(\alpha,\mu)=(0.2,8).
$$

È importante distinguere due concetti:

1. **non visto nel training** significa che questa coppia non ha contribuito alla matrice `snapshot_matrix`;
2. il FOM può comunque essere risolto esattamente nel nuovo punto, perché `simulate_allen_fom_implicit` riceve direttamente `alpha` e `mu`.

Nel passo implicito il residuo è

$$
F(x)
=
x-y^n
-\Delta t
\left[
0.2\,Lx
+
8(x-x^3)
\right].
$$

La traiettoria ottenuta viene salvata in `test_reference_traj_coarse` ed è il riferimento contro cui vengono confrontati i ROM.
#### Perche calcolare comunque il FOM nel punto di test

Il ROM deve essere confrontato con qualcosa di affidabile. Per questo, anche se $(0.2,8)$ non viene inserito nel training, si esegue una simulazione FOM separata con gli stessi parametri. Questa traiettoria e il **ground truth** del confronto.

Il confronto corretto e quindi

$$
Y_{\mathrm{ROM}}(0.2,8)
\quad\text{contro}\quad
Y_{\mathrm{FOM}}(0.2,8),
$$

non contro una media delle soluzioni di training.


In [ ]:
# ============================================================
# FOM DI TEST: NUOVA COPPIA PARAMETRICA (0.2, 8)
# ============================================================

# Qui alpha=0.2 e mu=8 entrano direttamente nell'equazione FOM.
# Questa simulazione NON viene usata per costruire snapshot_matrix.
test_reference_traj_coarse, test_fom_time = simulate_allen_fom_implicit(
    L,
    y0,
    dt,
    times,
    save,
    snapshot_times,
    test_alpha,
    test_mu,
)

print(f'Tempo FOM test: {test_fom_time:.6f} s')

## Addestramento e simulazione: POD e POD-DEIM

### POD-Galerkin e POD-DEIM nel caso parametrico

La base POD $\Psi$ viene costruita dalla matrice globale degli snapshot di training.

La rappresentazione ridotta è

$$
y(t)\approx \Psi a(t).
$$

Proiettando la dinamica di Allen–Cahn si ottiene

$$
\dot a
=
\alpha L_r a
+
\mu
\left[
a-\Psi^T(\Psi a)^3
\right],
\qquad
L_r=\Psi^TL\Psi.
$$

Questa formula spiega esattamente il comportamento parametrico del POD:

- $\Psi$ e $L_r$ vengono costruiti una volta durante il training;
- quando cambia $(\alpha,\mu)$, non si cambia la base;
- durante la simulazione online si sostituiscono direttamente i nuovi valori di $\alpha$ e $\mu$.

Per il test:

$$
\dot a
=
0.2\,L_r a
+
8
\left[
a-\Psi^T(\Psi a)^3
\right].
$$

POD-DEIM usa lo stesso principio. La sola differenza è che il termine cubico viene approssimato tramite i punti DEIM per ridurre il costo.
#### Derivazione pratica del POD-Galerkin

Si parte dall'approssimazione

$$
y\approx\Psi a.
$$

Sostituendola nel FOM e proiettando con $\Psi^T$ si ottiene

$$
\dot a
=
\alpha\Psi^TL\Psi a
+
\mu\left[a-\Psi^T(\Psi a)^3\right].
$$

Definendo una volta per tutte

$$
L_r=\Psi^TL\Psi,
$$

si arriva alla dinamica ridotta usata nel codice. La riduzione diminuisce il numero di variabili da 576 a `r_pod = 8`.

#### Perche serve DEIM

Nel POD standard la parte $(\Psi a)^3$ richiede ancora di ricostruire un vettore di dimensione $N=576$. DEIM introduce una seconda approssimazione della non linearita e seleziona `m_deim = 10` coordinate spaziali. In questo modo il termine cubico viene valutato soltanto nei punti DEIM e poi riportato nello spazio ridotto.

Quindi POD riduce principalmente la **dimensione dello stato**, mentre DEIM mira a ridurre anche il **costo della non linearita**.


In [ ]:
# ============================================================
# BLOCCO A — TRAINING DELLE BASI POD / POD-DEIM
# ============================================================

r_pod = 8
m_deim = 10

(
    Psi,
    Lr,
    sv_linear,
    nonlinear_projection,
    sampled_Psi,
    deim_idx,
    sv_nonlinear,
) = train_allen_pod(
    L,
    snapshot_matrix,
    r_pod,
    m_deim,
)

In [ ]:
# ============================================================
# BLOCCO B — ERRORI SUI 9 PUNTI DI TRAINING
# ============================================================

training_errors_pod = np.zeros(num_sims)
training_errors_deim = np.zeros(num_sims)

for k in range(num_sims):
    col_start = k * num_snaps_per_sim
    col_end = (k + 1) * num_snaps_per_sim

    # Traiettoria FOM esatta della coppia parametrica k.
    fom_train_esatto = snapshot_matrix[:, col_start:col_end]

    alpha_k = training_alphas[k]
    mu_k = training_mus[k]

    # I ROM vengono simulati usando gli stessi parametri del caso FOM.
    pod_tr, _ = simulate_allen_pod(
        Psi, Lr, y0,
        alpha_k, mu_k,
        dt, times, save
    )

    deim_tr, _ = simulate_allen_deim(
        Psi, Lr, nonlinear_projection, sampled_Psi,
        y0, alpha_k, mu_k,
        dt, times, save
    )

    training_errors_pod[k] = (
        np.linalg.norm(pod_tr - fom_train_esatto, ord='fro')
        / np.linalg.norm(fom_train_esatto, ord='fro')
    )

    training_errors_deim[k] = (
        np.linalg.norm(deim_tr - fom_train_esatto, ord='fro')
        / np.linalg.norm(fom_train_esatto, ord='fro')
    )

In [ ]:
# ============================================================
# BLOCCO C — GENERALIZZAZIONE AL PUNTO NON VISTO (0.2, 8)
# ============================================================

# La base rimane quella appresa dai 9 casi di training.
# Cambiano SOLO i coefficienti parametrici alpha e mu nella dinamica ridotta.
pod_traj, pod_time = simulate_allen_pod(
    Psi, Lr, y0,
    test_alpha, test_mu,
    dt, times, save
)

deim_traj, deim_time = simulate_allen_deim(
    Psi, Lr, nonlinear_projection, sampled_Psi,
    y0,
    test_alpha, test_mu,
    dt, times, save
)

# Confronto con il FOM risolto direttamente nello stesso punto parametrico.
pod_test_err = (
    np.linalg.norm(pod_traj - test_reference_traj_coarse, ord='fro')
    / np.linalg.norm(test_reference_traj_coarse, ord='fro')
)

deim_test_err = (
    np.linalg.norm(deim_traj - test_reference_traj_coarse, ord='fro')
    / np.linalg.norm(test_reference_traj_coarse, ord='fro')
)

print(f'Errore test POD      = {pod_test_err:.6e}')
print(f'Errore test POD-DEIM = {deim_test_err:.6e}')

## Costruzione basi Lift & Learn

### Costruzione del modello Lift & Learn parametrico

Si fissano

$$
r_y=4,\qquad r_z=4,
$$

quindi lo stato ridotto lifted ha dimensione totale

$$
r_{\mathrm{tot}}=r_y+r_z=8.
$$

Per ogni snapshot FOM di training:

1. si proietta $y$ sulla base `Psi_y`;
2. si costruisce $z=y^2$ e lo si proietta sulla base `Psi_z`;
3. si forma lo stato ridotto
   $$
   x=\begin{bmatrix}a_y\\a_z\end{bmatrix};
   $$
4. si calcola $q(x)$, cioè il vettore dei monomi quadratici;
5. si associa allo snapshot la coppia $(\alpha,\mu)$ che lo ha generato;
6. si costruiscono le feature
   $$
   \alpha x,\quad
   \mu x,\quad
   \alpha q(x),\quad
   \mu q(x).
   $$

La matrice `lift_F` è quindi la raccolta di tutte queste feature parametriche:

$$
F=
\begin{bmatrix}
\alpha x\\
\mu x\\
\alpha q(x)\\
\mu q(x)
\end{bmatrix}_{\text{per tutti gli snapshot}}.
$$

Il punto $(0.2,8)$ non serve per costruire `lift_F`: durante la simulazione online verranno semplicemente generate le nuove feature

$$
\begin{bmatrix}
0.2x\\
8x\\
0.2q(x)\\
8q(x)
\end{bmatrix}.
$$
#### Dati che entrano realmente nella regressione

Con `ry_lift = 4` e `rz_lift = 4`, lo stato lifted ridotto ha dimensione totale 8. Per ogni snapshot vengono memorizzati:

- le coordinate ridotte dello stato $a_y$;
- le coordinate ridotte del lifting $a_z$;
- la derivata ridotta $\dot x$;
- i valori di $\alpha$ e $\mu$ associati a quello snapshot.

Dopo aver costruito $Q=q(X)$, il dataset di regressione viene organizzato come

$$
F=
\begin{bmatrix}
\alpha X\\
\mu X\\
\alpha Q\\
\mu Q
\end{bmatrix},
\qquad
X_{\mathrm{dot}}=
\begin{bmatrix}
\dot x(t_1) & \cdots & \dot x(t_K)
\end{bmatrix}.
$$

L'apprendimento consiste nel trovare $C$ affinche $CF$ riproduca il piu possibile $X_{\mathrm{dot}}$ sui dati di training.


In [ ]:
# ============================================================
# COSTRUZIONE DELLE BASI E DELLE FEATURE LIFT & LEARN
# ============================================================

ry_lift = 4
rz_lift = 4

(
    Psi_y,
    Psi_z,
    sv_y,
    sv_z,
    lift_F,
    lift_Xdot,
    lift_masks,
) = build_allen_lift_data(
    L,
    training_alphas,
    training_mus,
    snapshot_matrix,
    num_snaps_per_sim,
    ry_lift,
    rz_lift,
)

print(f"Dimensione stato ridotto lifted: {ry_lift + rz_lift}")
print(f"Shape matrice feature lift_F: {lift_F.shape}")
print(f"Shape matrice derivate lift_Xdot: {lift_Xdot.shape}")

## Energia trattenuta al variare di r — modello Lifted

### Energia POD trattenuta al variare della dimensione della base lifted

Questa cella studia quanto rapidamente decadono i valori singolari dei due blocchi:

- $Y$, stato originale;
- $Z=Y^2$, stato sollevato.

Per i rank

$
r\in\{1,2,4,8,16,32\}
$

calcola la percentuale di energia trattenuta:

$
E_r
=
100\,
\frac{\sum_{i=1}^{r}\sigma_i^2}
{\sum_i\sigma_i^2}.
$

Il calcolo viene eseguito separatamente per `sv_y` e `sv_z`.

`min(r_curr, len(sv_*))` evita di richiedere più modi di quanti ne esistano effettivamente.

Il `DataFrame` finale mostra quindi, per ciascun valore di $r$, quanta energia degli snapshot viene catturata dalla corrispondente base POD.


In [ ]:

r_test_values = np.array([1, 2, 4, 8, 16, 32], dtype=int)
num_tests = len(r_test_values)

total_energy_y = np.sum(sv_y ** 2)
total_energy_z = np.sum(sv_z ** 2)

Energia_Y_perc = np.zeros(num_tests)
Energia_Z_perc = np.zeros(num_tests)

for i, r_curr in enumerate(r_test_values):
    r_curr_y = min(r_curr, len(sv_y))
    r_curr_z = min(r_curr, len(sv_z))

    Energia_Y_perc[i] = (
        np.sum(sv_y[:r_curr_y] ** 2) / total_energy_y
    ) * 100.0
    Energia_Z_perc[i] = (
        np.sum(sv_z[:r_curr_z] ** 2) / total_energy_z
    ) * 100.0

Tabella_EnergiaVsR = pd.DataFrame({
    'Dimensione_r': r_test_values,
    'Energia_Y_perc': Energia_Y_perc,
    'Energia_Z_perc': Energia_Z_perc,
})

print('\nENERGIA TRATTENUTA AL VARIARE DELLA BASE LIFTED\n')
display(Tabella_EnergiaVsR)


## Energia trattenuta e dimensione dei modelli

### Energia trattenuta dai tre modelli scelti

Questa cella riassume il contenuto energetico delle basi effettivamente utilizzate nel confronto.

#### POD-Galerkin
Per lo stato usa `r_pod = 8` modi. Non ha una base separata per la non linearità.

#### POD-DEIM
Usa:
- `r_pod = 8` modi POD per lo stato;
- `m_deim = 10` modi per rappresentare gli snapshot della non linearità cubica.

#### Lift & Learn
Usa:
- `ry_lift = 4` modi per $y$;
- `rz_lift = 4` modi per $z=y^2$.

Per ciascuna base l'energia viene calcolata come

$
100\,
\frac{\sum_{i=1}^{r}\sigma_i^2}
{\sum_i\sigma_i^2}.
$

La tabella permette quindi di leggere insieme **dimensione ridotta** ed **energia degli snapshot rappresentata** dai diversi approcci.


In [ ]:

en_pod_y = np.sum(sv_linear[:r_pod] ** 2) / np.sum(sv_linear ** 2)
en_deim_f = np.sum(sv_nonlinear[:m_deim] ** 2) / np.sum(sv_nonlinear ** 2)
en_lift_y = np.sum(sv_y[:ry_lift] ** 2) / np.sum(sv_y ** 2)
en_lift_z = np.sum(sv_z[:rz_lift] ** 2) / np.sum(sv_z ** 2)

Metodo_ROM = ['POD Galerkin', 'POD-DEIM', 'Lift & Learn']
Modi_Stato = [r_pod, r_pod, ry_lift]
Modi_NonLineari = [np.nan, m_deim, rz_lift]
Energia_Stato_Perc = [en_pod_y * 100, en_pod_y * 100, en_lift_y * 100]
Energia_NonLineare_Perc = [np.nan, en_deim_f * 100, en_lift_z * 100]

Tabella_ModelliScelti = pd.DataFrame({
    'Metodo_ROM': Metodo_ROM,
    'Modi_Stato': Modi_Stato,
    'Modi_NonLineari': Modi_NonLineari,
    'Energia_Stato_Perc': Energia_Stato_Perc,
    'Energia_NonLineare_Perc': Energia_NonLineare_Perc,
})

print('\nENERGIA TRATTENUTA DAI MODELLI SCELTI\n')
display(Tabella_ModelliScelti)


## Regolarizzazione e simulazione Lift & Learn

### Regolarizzazione e identificazione Lift & Learn

Per ogni valore candidato di regolarizzazione Tikhonov viene identificata una matrice di coefficienti $C$ che soddisfa, in senso ai minimi quadrati,

$$
\dot X
\approx
C F.
$$

Dopo il fit, il modello viene simulato sui punti parametrici di training per verificare che la dinamica rimanga finita.

Poi viene simulato anche nel punto

$$
(\alpha,\mu)=(0.2,8).
$$

Durante questa simulazione, a ogni passo temporale si costruiscono le feature

$$
\Theta_{\mathrm{test}}
=
\begin{bmatrix}
0.2x\\
8x\\
0.2q(x)\\
8q(x)
\end{bmatrix},
$$

e si calcola

$$
\dot x=C\,\Theta_{\mathrm{test}}.
$$

#### Nota metodologica importante

Nel codice attuale il punto $(0.2,8)$ viene usato anche per scegliere la regolarizzazione che minimizza l'errore.  
Perciò, in senso rigoroso, questo punto svolge anche il ruolo di **validation point** e non è un test completamente indipendente.

Il codice viene lasciato invariato per rispettare il workflow originale, ma il commento rende esplicita questa distinzione.
#### Che cosa sta scegliendo la regolarizzazione

Operator Inference e un problema di regressione. Se i coefficienti vengono adattati troppo aggressivamente ai dati, il modello puo diventare instabile o generalizzare male. La regolarizzazione di Tikhonov aggiunge una penalizzazione:

$$
\min_c\;\|Dc-y\|_2^2+\lambda\|c\|_2^2.
$$

Un valore piccolo di $\lambda$ privilegia l'adattamento ai dati; un valore maggiore limita la grandezza dei coefficienti. Il notebook prova diversi valori candidati e scarta i modelli che producono traiettorie non finite.

#### Nota sulla validazione

Nel workflow attuale il punto $(0.2,8)$ viene usato anche per scegliere la regolarizzazione migliore. Di conseguenza, per Lift & Learn questo punto e contemporaneamente un punto di selezione/validazione e di confronto finale. Per una valutazione rigorosamente indipendente sarebbe preferibile separare:

$$
\text{training}\;\longrightarrow\;\text{validation}\;\longrightarrow\;\text{test}.
$$

Questa nota non modifica il funzionamento del modello parametrico, ma e importante quando si interpreta la parola **generalizzazione**.


In [ ]:
# ============================================================
# BLOCCO A — VALORI CANDIDATI DI REGOLARIZZAZIONE
# ============================================================

candidate_regularizations = np.array([
    1e-10, 1e-9, 1e-8, 1e-7, 1e-6,
    1e-5, 1e-4, 1e-2, 1.0,
])

best_error = np.inf
best_regularization = np.nan
best_coefficients = None
best_lift_traj = None
best_lift_time = 0.0

In [ ]:
# ============================================================
# BLOCCO B — FIT + CONTROLLO STABILITÀ + SCELTA DEL MODELLO
# ============================================================

for reg in candidate_regularizations:
    # 1) Identificazione dei coefficienti usando SOLO i dati costruiti
    #    dalle nove simulazioni parametriche di training.
    coeff_candidate, _ = fit_allen_lift_coefficients(
        lift_F,
        lift_Xdot,
        lift_masks,
        reg,
    )

    # 2) Verifica che il modello sia stabile sui nove casi di training.
    training_stable = True

    for k in range(num_sims):
        train_traj_candidate, _ = simulate_allen_lift(
            y0,
            Psi_y,
            Psi_z,
            coeff_candidate,
            training_alphas[k],
            training_mus[k],
            dt,
            times,
            save,
        )

        if np.any(~np.isfinite(train_traj_candidate)):
            training_stable = False
            break

    if not training_stable:
        continue

    # 3) Simulazione nel punto parametrico (0.2, 8).
    #    Dentro simulate_allen_lift vengono costruite le feature:
    #    [0.2*x, 8*x, 0.2*q(x), 8*q(x)].
    test_traj_candidate, lift_time_candidate = simulate_allen_lift(
        y0,
        Psi_y,
        Psi_z,
        coeff_candidate,
        test_alpha,
        test_mu,
        dt,
        times,
        save,
    )

    if not np.all(np.isfinite(test_traj_candidate)):
        continue

    # 4) Errore rispetto al FOM di riferimento nello stesso punto parametrico.
    err_test = (
        np.linalg.norm(
            test_traj_candidate - test_reference_traj_coarse,
            ord='fro',
        )
        / np.linalg.norm(test_reference_traj_coarse, ord='fro')
    )

    # NOTA: questa scelta usa il punto di test per selezionare reg.
    # In una pipeline rigorosa servirebbe un punto di validation separato.
    if err_test < best_error:
        best_error = err_test
        best_regularization = reg
        best_coefficients = coeff_candidate
        best_lift_traj = test_traj_candidate
        best_lift_time = lift_time_candidate

if np.isnan(best_regularization):
    raise RuntimeError('Nessun modello stabile trovato.')

print(
    f'Regolarizzazione = {best_regularization:.1e} '
    f'(Errore Test = {best_error:.3e})'
)

lift_coefficients = best_coefficients
lift_test_err = best_error

In [ ]:
# ============================================================
# BLOCCO C — ERRORI LIFT & LEARN SUI PUNTI DI TRAINING
# ============================================================

training_errors_lift = np.zeros(num_sims)

for k in range(num_sims):
    col_start = k * num_snaps_per_sim
    col_end = (k + 1) * num_snaps_per_sim
    fom_train_esatto = snapshot_matrix[:, col_start:col_end]

    lift_tr, _ = simulate_allen_lift(
        y0,
        Psi_y,
        Psi_z,
        best_coefficients,
        training_alphas[k],
        training_mus[k],
        dt,
        times,
        save,
    )

    training_errors_lift[k] = (
        np.linalg.norm(lift_tr - fom_train_esatto, ord='fro')
        / np.linalg.norm(fom_train_esatto, ord='fro')
    )

## Sintesi: cosa viene costruito offline e cosa cambia online

| Metodo | Costruito offline | Dipende dal nuovo $(\alpha,\mu)$ online | Stato integrato |
|---|---|---|---|
| POD | $\Psi$, $L_r$ | coefficienti $\alpha$, $\mu$ nella dinamica ridotta | $a\in\mathbb{R}^{8}$ |
| POD-DEIM | $\Psi$, $L_r$, base/indici DEIM, operatore DEIM | coefficienti $\alpha$, $\mu$ | $a\in\mathbb{R}^{8}$ |
| Lift & Learn | $\Psi_y$, $\Psi_z$, maschere, coefficienti $C$ | feature $[\alpha x,\mu x,\alpha q(x),\mu q(x)]^T$ | principalmente $a_y$, con $a_z$ ricostruito dal lifting |

Il punto concettuale centrale e quindi:

> **il training costruisce una rappresentazione e/o una legge dinamica ridotta; la fase online inserisce nuovi parametri in quella legge e la integra nel tempo.**

Non viene effettuata un'interpolazione diretta tra le nove traiettorie FOM di training.


## Tabelle riassuntive finali

### Tabelle finali

Le tabelle sono separate in tre blocchi per distinguere chiaramente:

1. prestazioni sul punto parametrico $(0.2,8)$;
2. errori sui nove punti di training;
3. confronto tra errore medio/massimo di training e errore sul punto non presente nel training.

Ricorda che, nel workflow attuale, Lift & Learn usa $(0.2,8)$ anche nella scelta della regolarizzazione.
#### Come leggere i risultati

Il confronto finale deve rispondere a due domande:

1. **Accuratezza:** quanto il ROM si discosta dal FOM?
2. **Efficienza:** quanto tempo si risparmia rispetto al FOM?

L'errore relativo usato e

$$
e_{\mathrm{rel}}=
\frac{\|Y_{\mathrm{ROM}}-Y_{\mathrm{FOM}}\|_F}
{\|Y_{\mathrm{FOM}}\|_F}.
$$

Lo speedup e

$$
S=\frac{t_{\mathrm{FOM}}}{t_{\mathrm{ROM}}}.
$$

Un buon ROM deve mantenere l'errore contenuto e, allo stesso tempo, avere uno speedup significativamente maggiore di 1.


In [ ]:
# ============================================================
# TABELLA 1 — RISULTATI NEL PUNTO PARAMETRICO (0.2, 8)
# ============================================================

Modello = ['FOM Esatto', 'POD', 'POD-DEIM', 'Lift & Learn']
Errore_Relativo = np.array([
    0.0,
    pod_test_err,
    deim_test_err,
    lift_test_err,
])
Tempo_Secondi = np.array([
    test_fom_time,
    pod_time,
    deim_time,
    best_lift_time,
])
Speedup = test_fom_time / Tempo_Secondi

Tabella_test = pd.DataFrame({
    'Modello': Modello,
    'Errore_Relativo': Errore_Relativo,
    'Tempo_Secondi': Tempo_Secondi,
    'Speedup': Speedup,
})

print('\nTABELLA RISULTATI TEST\n')
display(Tabella_test)

In [ ]:
# ============================================================
# TABELLA 2 — ERRORI SUI 9 PUNTI PARAMETRICI DI TRAINING
# ============================================================

Tabella_Training = pd.DataFrame({
    'Alpha_Training': training_alphas,
    'Mu_Training': training_mus,
    'Errore_POD': training_errors_pod,
    'Errore_POD_DEIM': training_errors_deim,
    'Errore_Lift_Learn': training_errors_lift,
})

print('\nTABELLA ERRORE SUI PUNTI DI TRAINING\n')
display(Tabella_Training)

In [ ]:
# ============================================================
# TABELLA 3 — TRAINING vs PUNTO PARAMETRICO NON PRESENTE NEL TRAINING
# ============================================================

Metodo_Confronto = ['POD Galerkin', 'POD-DEIM', 'Lift & Learn']

Errore_Training_Medio = np.array([
    np.mean(training_errors_pod),
    np.mean(training_errors_deim),
    np.mean(training_errors_lift),
])

Errore_Training_Max = np.array([
    np.max(training_errors_pod),
    np.max(training_errors_deim),
    np.max(training_errors_lift),
])

Errore_Test_NonVisto = np.array([
    pod_test_err,
    deim_test_err,
    lift_test_err,
])

Tabella_RiepilogoTraining = pd.DataFrame({
    'Metodo_Confronto': Metodo_Confronto,
    'Errore_Training_Medio': Errore_Training_Medio,
    'Errore_Training_Max': Errore_Training_Max,
    'Errore_Test_NonVisto': Errore_Test_NonVisto,
})

print('\nRIEPILOGO TRAINING vs PUNTO NON VISTO\n')
display(Tabella_RiepilogoTraining)

## Grafici

### Grafici

Anche i grafici sono separati in blocchi distinti:

1. evoluzione temporale al centro del dominio;
2. confronto del campo finale FOM / Lift & Learn;
3. errore assoluto finale Lift & Learn;
4. decadimento dei valori singolari.

Tutti i confronti dinamici di questa sezione si riferiscono al punto parametrico

$$
(\alpha,\mu)=(0.2,8).
$$

In [ ]:
# ============================================================
# GRAFICO 1 — EVOLUZIONE TEMPORALE AL CENTRO DEL DOMINIO
# ============================================================

# MATLAB center_i = floor(n/2)+1 (1-based)
# Python usa indicizzazione 0-based.
center_i = n // 2
center_index = center_i * n + center_i

line_styles = ['k-', 'b--', 'g:', 'r-.']

plt.figure(figsize=(9, 4.5))

plt.plot(
    snapshot_times,
    test_reference_traj_coarse[center_index, :],
    line_styles[0],
    label=Modello[0],
    linewidth=1.5,
)

plt.plot(
    snapshot_times,
    pod_traj[center_index, :],
    line_styles[1],
    label=Modello[1],
    linewidth=1.5,
)

plt.plot(
    snapshot_times,
    deim_traj[center_index, :],
    line_styles[2],
    label=Modello[2],
    linewidth=2.0,
)

plt.plot(
    snapshot_times,
    best_lift_traj[center_index, :],
    line_styles[3],
    label=Modello[3],
    linewidth=1.5,
)

plt.xlabel('Tempo [s]')
plt.ylabel('y(0.5, 0.5, t)')
plt.title(
    rf'Evoluzione Centro Dominio '
    rf'($\alpha={test_alpha:g}$, $\mu={test_mu:g}$)'
)
plt.legend(loc='best')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# GRAFICO 2 — CAMPO FINALE: FOM vs LIFT & LEARN
# ============================================================

fom_final = (
    test_reference_traj_coarse[:, -1]
    .reshape((n, n), order='F')
    .T
)

lift_final = (
    best_lift_traj[:, -1]
    .reshape((n, n), order='F')
    .T
)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))

im0 = axes[0].imshow(
    fom_final,
    extent=[0, 1, 0, 1],
    origin='lower',
    aspect='equal',
)
axes[0].set_xlabel(r'$x_1$')
axes[0].set_ylabel(r'$x_2$')
axes[0].set_title('Soluzione FOM finale')
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(
    lift_final,
    extent=[0, 1, 0, 1],
    origin='lower',
    aspect='equal',
)
axes[1].set_xlabel(r'$x_1$')
axes[1].set_ylabel(r'$x_2$')
axes[1].set_title('Ricostruzione Lift & Learn finale')
fig.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# GRAFICO 3 — ERRORE ASSOLUTO FINALE LIFT & LEARN
# ============================================================

final_error = np.abs(
    best_lift_traj[:, -1]
    - test_reference_traj_coarse[:, -1]
)

final_error_field = (
    final_error
    .reshape((n, n), order='F')
    .T
)

plt.figure(figsize=(6, 5))
im = plt.imshow(
    final_error_field,
    extent=[0, 1, 0, 1],
    origin='lower',
    aspect='equal',
)

plt.colorbar(im)
plt.xlabel(r'$x_1$')
plt.ylabel(r'$x_2$')
plt.title('Errore assoluto finale Lift & Learn')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# GRAFICO 4 — DECADIMENTO DEI VALORI SINGOLARI
# ============================================================

sv_y_norm = sv_y / sv_y[0]
sv_z_norm = sv_z / sv_z[0]

num_plot = min(20, len(sv_y_norm), len(sv_z_norm))
mode_idx = np.arange(1, num_plot + 1)

plt.figure(figsize=(8, 4.8))

plt.semilogy(
    mode_idx,
    sv_y_norm[:num_plot],
    'b-',
    label='Valori singolari (blocco y)',
    linewidth=2,
)

plt.semilogy(
    mode_idx,
    sv_z_norm[:num_plot],
    'r--',
    label=r'Valori singolari (blocco z = $y^2$)',
    linewidth=2,
)

plt.xlabel('Indice del modo (i)')
plt.ylabel(r'$\sigma_i / \sigma_1$')
plt.title('Decadimento dei valori singolari')
plt.legend(loc='upper right')
plt.grid(True)
plt.tight_layout()
plt.show()

### Confronto grafico degli errori sui dati di training

Questa cella crea un grafico a barre raggruppate per le nove combinazioni di training.

Per ogni coppia $(\alpha,\mu)$ vengono mostrati affiancati gli errori relativi di:

- POD;
- POD-DEIM;
- Lift & Learn.

L'asse verticale usa una scala logaritmica, utile perché gli errori dei diversi metodi possono differire di più ordini di grandezza.

Le etichette dell'asse orizzontale riportano direttamente i valori di $\alpha$ e $\mu$, così il grafico permette di individuare in quali regioni dello spazio parametrico ciascun ROM risulta più o meno accurato.


In [ ]:

training_labels = [
    rf'$\alpha={a:g}$\n$\mu={m:g}$'
    for a, m in zip(training_alphas, training_mus)
]
bar_data = np.column_stack((
    training_errors_pod,
    training_errors_deim,
    training_errors_lift,
))

x_pos = np.arange(num_sims)
width = 0.25

plt.figure(figsize=(9, 4.8))
plt.bar(x_pos - width, bar_data[:, 0], width=width, label='POD')
plt.bar(x_pos,         bar_data[:, 1], width=width, label='POD-DEIM')
plt.bar(x_pos + width, bar_data[:, 2], width=width, label='Lift & Learn')
plt.yscale('log')
plt.xticks(x_pos, training_labels)
plt.xlabel('Combinazione di addestramento')
plt.ylabel('Errore relativo (scala log)')
plt.title('Errore sui dati di training: POD vs POD-DEIM vs Lift & Learn')
plt.legend(loc='best')
plt.grid(True)
plt.tight_layout()
plt.show()
